# 03 — Gold Aggregate

## Story beat: "Publish the data product"

Executives don't query `fact_sales` row by row. They ask: *"Which region sold the most yesterday?"* and *"Which category is trending?"* The **gold** layer answers those questions with **pre-aggregated, well-named tables** — the **data products** your domain shares with the rest of the organization (and with Power BI in Module 4).

---

## Gold tables we create

| Table | Grain | Measures | Consumers |
| --- | --- | --- | --- |
| `gold.sales_by_store_day` | One row per store per day | net_sales, units, transactions | Regional dashboards, Direct Lake model |
| `gold.sales_by_category` | One row per product category | net_sales, units | Category mix reports |

Both are **V-Ordered** and **OPTIMIZE**d — same pattern as silver.

> **Data mesh angle (Module 6):** Gold tables are what you'd register as domain-owned **data products** in a Retail domain — shared via Shortcuts to Finance or Operations domains.

In [ ]:
%run 00_setup

## Step 1 — Load silver tables

We read from silver (not bronze) — gold always builds on trusted, conformed data.

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.vorder.default", "true")

fact  = spark.table(f"{SILVER_SCHEMA}.fact_sales")
store = spark.table(f"{SILVER_SCHEMA}.dim_store")
prod  = spark.table(f"{SILVER_SCHEMA}.dim_product")

## Step 2 — Aggregate and write gold tables

**`sales_by_store_day`** — joins store attributes (region, city) so Power BI doesn't need extra relationships for regional charts.

**`sales_by_category`** — joins product dimension for category labels; one row per category for pie/bar visuals.

In [ ]:
sales_by_store_day = (
    fact.groupBy("sale_date", "store_id")
    .agg(
        F.sum("net_amount").alias("net_sales"),
        F.sum("quantity").alias("units"),
        F.countDistinct("sale_id").alias("transactions"),
    )
    .join(store, "store_id")
)
sales_by_store_day.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_SCHEMA}.sales_by_store_day")

sales_by_category = (
    fact.join(prod, "product_id")
    .groupBy("category")
    .agg(F.sum("net_amount").alias("net_sales"), F.sum("quantity").alias("units"))
)
sales_by_category.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_SCHEMA}.sales_by_category")
print("gold tables written: sales_by_store_day, sales_by_category")

## Step 3 — V-Order + preview

These tables feed **Module 4 Direct Lake** — layout matters. After OPTIMIZE, open the lakehouse SQL endpoint and run:

```sql
SELECT * FROM gold.sales_by_category ORDER BY net_sales DESC;
```

In [ ]:
for t in [f"{GOLD_SCHEMA}.sales_by_store_day", f"{GOLD_SCHEMA}.sales_by_category"]:
    spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')")
    spark.sql(f"OPTIMIZE {t}")

display(spark.sql(f"SELECT * FROM {GOLD_SCHEMA}.sales_by_category ORDER BY net_sales DESC"))

---

## ✅ Success looks like

- Two gold tables in the lakehouse Explorer.
- Category ranking displays (e.g. Electronics vs Grocery).

**Next:** `04_vorder_demo` (prove V-Order) → `05_shortcuts` (federation) → Module 2 warehouse.